In [ ]:
import json
from pathlib import Path
from evaluator import BenchmarkEvaluator

# Inizializziamo l'evaluator che contiene la logica per gestire la struttura dei tuoi JSON
evaluator = BenchmarkEvaluator()

# Definizione dei percorsi di input e output
answers_base = Path("/home/dario/miniconda3/envs/denv/Personas-Effect-On-Factual-Knowledge-Retrieval/answers")
metrics_base = Path("/home/dario/miniconda3/envs/denv/Personas-Effect-On-Factual-Knowledge-Retrieval/metrics")

# Iterazione ricorsiva su tutti i file .json presenti nella cartella answers
for input_path in answers_base.glob("*/*/*.json"):
    # Estraiamo dataset, modello e nome del file preservando la struttura ad albero
    dataset = input_path.parts[-3]
    model = input_path.parts[-2]
    filename = input_path.name

    # Costruiamo il percorso di destinazione finale
    output_path = metrics_base / dataset / model / filename
    
    # Se il file delle metriche esiste già, saltiamo l'elaborazione per ottimizzare i tempi
    if output_path.exists():
        continue

    print(f"Elaborazione metriche per: {output_path}")

    # Creiamo le cartelle di destinazione se non sono già esistenti
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Carichiamo il file con la struttura a 5 run generato dallo script precedente
    with input_path.open(encoding="utf-8") as f:
        runs_data = json.load(f)

    try:
        # Calcoliamo l'array dei 5 aggregatori (uno per ogni run)
        aggregated_metrics = evaluator.evaluate_json_results(runs_data)
        
        # Scriviamo il file di output finale mantenendo la formattazione pulita
        output_path.write_text(
            json.dumps(aggregated_metrics, indent=2, ensure_ascii=False) + "\n", 
            encoding="utf-8"
        )
    except Exception as e:
        print(f"Errore durante l'elaborazione del file {input_path}: {e}")